## Preliminaries

This code requires a modified version of the [`terratorch` repository](https://github.com/drobiu/terratorch/tree/fixlip)

Additionally, we will use the `plotting_utils.py` file from the `Terramind` repository for easier plotting

In [ ]:
import sys

import matplotlib.pyplot as plt
import src

import torch
import torch.nn.functional as F
import rioxarray as rxr
import matplotlib.pyplot as plt
from plotting_utils import s2_to_rgb, plot_s2, lulc_to_rgb, s1_to_rgb, plot_s1
from terratorch.registry import FULL_MODEL_REGISTRY
from glob import glob

# Select device
if torch.cuda.is_available():
    device = 'cuda'    
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

## Data loading

Here, we're loading the data for a sample from the [`sen1floods11`](https://github.com/cloudtostreet/Sen1Floods11) dataset. If needed, download the dataset using `gsutil -m rsync -r gs://sen1floods11 /YOUR/LOCAL/DIRECTORY/HERE`.

The images will be cropped to the TerraMind input size of 224x224.

In [ ]:
classes = ['unknown', 'water', 'trees', 'flooded vegetation', 'crops', 'built area', 'bare ground', 'snow/ice', 'clouds', 'rangeland']
l = 0
s = 224
img_s2 = '../data/sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/Bolivia_129334_S2Hand.tif'
img_s1 = '../data/sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/Bolivia_129334_S1Hand.tif'
data_s2 = rxr.open_rasterio(img_s2).values[:, l:l+s, l:l+s]
data_s1 = rxr.open_rasterio(img_s1).values[:, l:l+s, l:l+s]
label = rxr.open_rasterio('../data/sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand/Bolivia_23014_LabelHand.tif').values[:, l:l+s, l:l+s]

input_image_s2 = torch.tensor(data_s2, dtype=torch.float, device=device).unsqueeze(0)
input_image_s1 = torch.tensor(data_s1, dtype=torch.float, device=device).unsqueeze(0)
input_text = torch.tensor([[-20, 15]]).to(device)

In [ ]:
# Visualize the input
plt.imshow(s2_to_rgb(input_image_s2.detach().cpu().numpy()))
plt.show()
plt.imshow(s1_to_rgb(input_image_s1.detach().cpu().numpy()))

## Initialize the model

We will initialize a model for predicting LULC classes from the multimodal Sentinel-1 and Sentinel-2 input.

In [ ]:
# Define the output from S2L2A, S1GRD, S1RTC, DEM, LULC, and NDVI
output_modalities = ['LULC']

# Build model
model = FULL_MODEL_REGISTRY.build(
    'terramind_v1_small_generate',
    modalities=['S2L1C', 'S1GRD'],
    output_modalities=output_modalities,
    pretrained=True,
    standardize=True,
    timesteps=100,  # Number of diffusion steps
)

model = model.to(device)
lulc = model({'S2L1C': input_image_s2.repeat(4, 1, 1, 1), 'S1GRD': input_image_s1.repeat(4, 1, 1, 1)})['LULC']

In [ ]:
# Visualize LULC prediction
plt.imshow(lulc_to_rgb(lulc.mean(0).detach().cpu().numpy()))
F.one_hot(lulc.argmax(1)).float().mean((0, 1, 2)).cpu().numpy()

## Initialize the explainer

We will explain the model output with regard to class 1 - water

We set `grid_step` to 7, essentially aggregating tokens to a 2x2 grid, making the computation quicker

In [ ]:
# Select the class to explain
cl = 1

game = src.game_terramind.VisionLanguageGame(
    model=model,
    inputs={'S2L1C': input_image_s2, 'S1GRD': input_image_s1}, 
    modalities={'S2L1C': 'image', 'S1GRD': 'image'}, 
    mask_names={'S2L1C': 'untok_sen2l1c@224', 'S1GRD': 'untok_sen1grd@224'},
    batch_size=16,
    cl=cl,
    grid_step=7,
)

fixlip = src.fixlip.FIxLIP(
    n_players_modalities=game.n_players_image | game.n_players_text, 
    max_order=2,
    p=0.5, # weight
    mode="banzhaf",
    random_state=0
)

# Print the interactions
src.utils.set_seed(0)
interaction_values = fixlip.approximate_crossmodal(game, budget=2**15)
print(interaction_values)

In [ ]:
n_players_image = game.n_players_image
n_players_text = game.n_players_text
# fig, axs = plt.subplots(1, 2, figsize=(12, 4))
fig = plt.figure(figsize=(12, 8))
ax1 = plt.subplot(2,3,1)
ax2 = plt.subplot(2,3,2)
ax3 = plt.subplot(2,3,4)
ax4 = plt.subplot(2,3,5)
ax5 = plt.subplot(2,3,6)
axs = [ax1, ax2, ax3, ax4, ax5]


orig = {'S2L1C': torch.tensor(data_s2, dtype=torch.float, device=device).unsqueeze(0), 'S1GRD': torch.tensor(data_s1, dtype=torch.float, device=device).unsqueeze(0)}
# orig = {'S2L1C': torch.tensor(data_s2, dtype=torch.float, device=device).unsqueeze(0), 'Coords': input_text}
tokenized = model.forward_tokenizer(orig)
out = model.forward_tokenized(tokenized, orig)

src.plot.plot_image_and_text_together(
    img={'S2L1C': s2_to_rgb(input_image_s2), 'S1GRD': s1_to_rgb(input_image_s1)},
    # text=tokenized['coords']['tensor'].cpu().detach().numpy()[0].astype(str)[:-1],
    # text=input_text.cpu().detach().numpy()[0].astype(str),
    text=[],
    image_players=n_players_image,
    iv=interaction_values,
    plot_interactions=True,
    top_k=50,
    normalize_jointly=True,
    figsize=(7, 7),
    fontsize=12,
    margin=0.3,
    color_text=True,
    plot_heatmap=True,
    show=True,
    max_value=None,
    axes=axs[:2]
)
# fig.suptitle(f"Attributions class {classes[cl]}")
# plt.show()
# fig, axs = plt.subplots(1, 3, figsize=(12, 4))

# out['S2L2A'].shape
# out = model(orig)
axs[4].imshow(lulc_to_rgb(lulc.mean(0).detach().cpu().numpy()))
axs[4].axis('off')
axs[4].set_title("Original segmentation")

plot_s2(data_s2, ax=axs[2])
axs[2].set_title("Original image S2")
# plt.savefig(f'figures/attributions/attribution_{img_s2.split("/")[-1][:-4]}_class_{cl}')

plot_s1(data_s1, ax=axs[3])
axs[3].set_title("Original image S1")
# plt.savefig(f'figures/attributions_multiimage/attribution_{img_s2.split("/")[-1][:-4]}_class_{cl}')
plt.tight_layout()
plt.show()

## Compute explanations for multiple samples

In [ ]:
plt.ioff()
for fname in glob('data/sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/*_S2Hand.tif'):

    img_s2 = fname
    try:
        img_s1 = f'data/sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/{fname.split("/")[-1][:-11]}_S1Hand.tif'
        data_s2 = rxr.open_rasterio(img_s2).values[:, l:l+s, l:l+s]
        data_s1 = rxr.open_rasterio(img_s1).values[:, l:l+s, l:l+s]

        input_image_s2 = torch.tensor(data_s2, dtype=torch.float, device=device).unsqueeze(0)
        input_image_s1 = torch.tensor(data_s1, dtype=torch.float, device=device).unsqueeze(0)
        lulc = model({'S2L1C': input_image_s2.repeat(4, 1, 1, 1), 'S1GRD': input_image_s1.repeat(4, 1, 1, 1)})['LULC']
        cl_parts = F.one_hot(lulc.argmax(1)).float().mean((0, 1, 2)).cpu().numpy()
    except:
        continue

    for i, cl in enumerate(cl_parts):
        if i == 0 or cl < 0.1 or cl > 0.5:
            continue

        try:
            game = src.game_terramind.VisionLanguageGame(
                model=model,
                inputs={'S2L1C': input_image_s2, 'S1GRD': input_image_s1}, 
                modalities={'S2L1C': 'image', 'S1GRD': 'image'}, 
                mask_names={'S2L1C': 'untok_sen2l1c@224', 'S1GRD': 'untok_sen1grd@224'},
                batch_size=16,
                cl=i,
                grid_step=7,
            )

            fixlip = src.fixlip.FIxLIP(
                n_players_modalities=game.n_players_image | game.n_players_text, 
                max_order=2,
                p=0.5, # weight
                mode="banzhaf",
                random_state=0
            )

            src.utils.set_seed(0)
            interaction_values = fixlip.approximate_crossmodal(game, budget=2**13)
        except:
            continue

        n_players_image = game.n_players_image
        n_players_text = game.n_players_text

        # fig, axs = plt.subplots(1, 2, figsize=(12, 4))
        fig, axs = plt.subplots(2, 3, figsize=(12, 8))
        axs = axs.flatten()


        orig = {'S2L1C': torch.tensor(data_s2, dtype=torch.float, device=device).unsqueeze(0), 'S1GRD': torch.tensor(data_s1, dtype=torch.float, device=device).unsqueeze(0)}
        # orig = {'S2L1C': torch.tensor(data_s2, dtype=torch.float, device=device).unsqueeze(0), 'Coords': input_text}
        tokenized = model.forward_tokenizer(orig)
        out = model.forward_tokenized(tokenized, orig)

        src.plot.plot_image_and_text_together(
            img={'S2L1C': s2_to_rgb(input_image_s2), 'S1GRD': s1_to_rgb(input_image_s1)},
            # text=tokenized['coords']['tensor'].cpu().detach().numpy()[0].astype(str)[:-1],
            # text=input_text.cpu().detach().numpy()[0].astype(str),
            text=[],
            image_players=n_players_image,
            iv=interaction_values,
            plot_interactions=True,
            top_k=50,
            normalize_jointly=True,
            figsize=(7, 7),
            fontsize=12,
            margin=0.3,
            color_text=True,
            plot_heatmap=True,
            show=True,
            max_value=None,
            axes=axs[:2]
        )

        sample_name = img_s2.split("/")[-1][:-4]

        fig.suptitle(f"File: {sample_name}, Class: {classes[i]}")

        axs[2].text(0, 0, str(interaction_values), wrap=True)
        axs[2].axis('off')

        plot_s2(data_s2, ax=axs[3])
        axs[3].set_title("Original image S2")

        plot_s1(data_s1, ax=axs[4])
        axs[4].set_title("Original image S1")
        
        axs[5].imshow(lulc_to_rgb(lulc.mean(0).detach().cpu().numpy()))
        axs[5].axis('off')
        axs[5].set_title("Original segmentation")

        plt.tight_layout()
        plt.savefig(f'figures/attributions_multiimage/attribution_{sample_name}_class_{i}')


# game.empty_value, game.full_value

In [ ]:
plt.ioff()
for fname in glob('data/sen1floods11/v1.1/data/flood_events/HandLabeled/S2Hand/*_S2Hand.tif'):

    img_s2 = fname
    try:
        img_s1 = f'data/sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand/{fname.split("/")[-1][:-11]}_S1Hand.tif'
        data_s2 = rxr.open_rasterio(img_s2).values[:, l:l+s, l:l+s]
        data_s1 = rxr.open_rasterio(img_s1).values[:, l:l+s, l:l+s]

        input_image_s2 = torch.tensor(data_s2, dtype=torch.float, device=device).unsqueeze(0)
        input_image_s1 = torch.tensor(data_s1, dtype=torch.float, device=device).unsqueeze(0)
        lulc = model({'S2L1C': input_image_s2.repeat(4, 1, 1, 1), 'S1GRD': input_image_s1.repeat(4, 1, 1, 1)})['LULC']
        cl_parts = F.one_hot(lulc.argmax(1)).float().mean((0, 1, 2)).cpu().numpy()
    except:
        continue

    for i, cl in enumerate(cl_parts):
        if i == 0 or cl < 0.1 or cl > 0.5:
            continue

        try:
            game = src.game_terramind.VisionLanguageGame(
                model=model,
                inputs={'S2L1C': input_image_s2, 'S1GRD': input_image_s1}, 
                modalities={'S2L1C': 'image', 'S1GRD': 'image'}, 
                mask_names={'S2L1C': 'untok_sen2l1c@224', 'S1GRD': 'untok_sen1grd@224'},
                batch_size=16,
                cl=i,
                grid_step=4,
            )

            fixlip = src.fixlip.FIxLIP(
                n_players_modalities=game.n_players_image | game.n_players_text, 
                max_order=2,
                p=0.5, # weight
                mode="banzhaf",
                random_state=0
            )

            src.utils.set_seed(0)
            interaction_values = fixlip.approximate_crossmodal(game, budget=2**15)
        except:
            continue

        n_players_image = game.n_players_image
        n_players_text = game.n_players_text

        # fig, axs = plt.subplots(1, 2, figsize=(12, 4))
        fig, axs = plt.subplots(2, 3, figsize=(12, 8))
        axs = axs.flatten()


        orig = {'S2L1C': torch.tensor(data_s2, dtype=torch.float, device=device).unsqueeze(0), 'S1GRD': torch.tensor(data_s1, dtype=torch.float, device=device).unsqueeze(0)}
        # orig = {'S2L1C': torch.tensor(data_s2, dtype=torch.float, device=device).unsqueeze(0), 'Coords': input_text}
        tokenized = model.forward_tokenizer(orig)
        out = model.forward_tokenized(tokenized, orig)

        src.plot.plot_image_and_text_together(
            img={'S2L1C': s2_to_rgb(input_image_s2), 'S1GRD': s1_to_rgb(input_image_s1)},
            # text=tokenized['coords']['tensor'].cpu().detach().numpy()[0].astype(str)[:-1],
            # text=input_text.cpu().detach().numpy()[0].astype(str),
            text=[],
            image_players=n_players_image,
            iv=interaction_values,
            plot_interactions=True,
            top_k=50,
            normalize_jointly=True,
            figsize=(7, 7),
            fontsize=12,
            margin=0.3,
            color_text=True,
            plot_heatmap=True,
            show=True,
            max_value=None,
            axes=axs[:2]
        )

        sample_name = img_s2.split("/")[-1][:-4]

        fig.suptitle(f"File: {sample_name}, Class: {classes[i]}")

        axs[2].text(0, 0, str(interaction_values), wrap=True)
        axs[2].axis('off')

        plot_s2(data_s2, ax=axs[3])
        axs[3].set_title("Original image S2")

        plot_s1(data_s1, ax=axs[4])
        axs[4].set_title("Original image S1")
        
        axs[5].imshow(lulc_to_rgb(lulc.mean(0).detach().cpu().numpy()))
        axs[5].axis('off')
        axs[5].set_title("Original segmentation")

        plt.tight_layout()
        plt.savefig(f'figures/attributions_multiimage/attribution_dense_{sample_name}_class_{i}')


# game.empty_value, game.full_value

In [ ]:
interaction_values.interactions

In [ ]:
inter = {}
c = 48
for k, v in interaction_values.interaction_lookup.items():
    if len(k) < 2:
        continue
    if (k[0] > c and k[1] < c) or (k[0] < c and k[1] > c):
        inter[k] = interaction_values.values[v]

In [ ]:
{k: v for k, v in sorted(inter.items(), key=lambda item: item[1])}

In [ ]:
plt.imshow(lulc_to_rgb(game.outs[0][6]))

In [ ]:
tokenized['coords']['tensor'].cpu().detach().numpy()[0].astype(str)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12, 4))


out = model(orig)
orig = {'S2L1C': input_image, 'Coords': input_text}
tokenized = model.forward_tokenizer(orig)
out = model.forward_tokenized(tokenized, orig)
# out = model(orig)
axs[1].imshow(lulc_to_rgb(out['LULC'].detach().cpu()))
plot_s2(data, ax=axs[2])
plt.show()

In [ ]:
from copy import deepcopy
# input_image[:, :, :100, :100] = 0
orig = {'S2L1C': input_image_s2, 'S1GRD': input_image_s1}

# mask = torch.ones(1, 196, dtype=torch.bool, device=device)
mask = game.masks[0]['untok_sen2l1c@224'][[0]]
mask_coords = torch.ones(1, 2, dtype=torch.bool, device=device)
masks = {'untok_sen2l1c@224': mask, 'coords': mask_coords}
# mask[0, 20:180] = False
model.forward_tokenizer(deepcopy(orig))
fig, axs = plt.subplots(1, 2, figsize=(14, 7))
axs[0].imshow(lulc_to_rgb(model.forward(orig, mask=masks)['LULC'].cpu().detach().numpy()))
axs[1].imshow(mask.cpu().numpy().reshape(14, 14))
print(mask)

In [ ]:
len(game.outs)

In [ ]:
mask.logical_not()

In [ ]:
(out['LULC'].argmax(axis=1) == ).to(torch.float).mean((1, 2))

In [ ]:
game.masks[0]['untok_sen2l1c@224'][0]

In [ ]:
fixlip.sampler_image.coalitions_matrix.shape

In [ ]:
out = model({'S2L1C': input_image, 'Coords': input_text})
lulc = out['LULC']
plt.imshow(lulc_to_rgb(lulc.cpu().detach().numpy()))

In [ ]:
lulc.softmax(1)[:, 8].mean()

In [ ]:
lulc = lulc.argmax(1)
lulc[lulc == 1] = 2
plt.imshow(lulc.cpu().detach().numpy()[0, :, :])

In [ ]:
model = FULL_MODEL_REGISTRY.build(
    'terramind_v1_tiny_generate',
    modalities=['S2L1C'],
    output_modalities=['S2L1C'],
    pretrained=True,
    standardize=True,
    timesteps=10,  # Number of diffusion steps
)

model = model.to(device)


In [ ]:
from copy import deepcopy
# input_image[:, :, :100, :100] = 0
orig = {'S2L1C': input_image_s2, 'S1GRD': input_image_s1}

mask_s2 = torch.ones(1, 196, dtype=torch.bool, device=device)
mask_s1 = torch.ones(1, 196, dtype=torch.bool, device=device)
masks = {'untok_sen2l1c@224': mask_s2, 'untok_sen1grd@224': mask_s1}
mask_s1[0, 0:100] = False
# model.forward_tokenizer(deepcopy(orig))
fig, axs = plt.subplots(1, 5, figsize=(14, 7))
axs[0].imshow(lulc_to_rgb(model.forward(orig, mask=masks)['LULC'].cpu().detach().numpy()))
plot_s2(data_s2, ax=axs[1])
axs[2].imshow(mask_s2.cpu().numpy().reshape(14, 14))
plot_s1(data_s1, ax=axs[3])
axs[4].imshow(mask_s1.cpu().numpy().reshape(14, 14))
print(mask_s1)

In [ ]:
help(model)